Revision groups -> sentence json

- reads every article from `data/input/revision_groups.jsonl.gz` (one article per line)
- splits every version (paragraph) into sentences with spaCy `en_core_web_trf` (parser sentence boundaries) - the same
  parser the feature-extraction notebooks use, so a sentence is defined the same way everywhere
- writes one json per article to `data/sentence/sentence-json`, but every version becomes `{"paragraph": ..., "sentences": [...]}`

In [1]:
from pathlib import Path
import spacy

revision_groups_file = Path("../../data/input/revision_groups.jsonl.gz")  # one article per line
sentence_json_path = Path("../../data/sentence/sentence-json")

sentence_json_path.mkdir(parents=True, exist_ok=True)

# en_core_web_trf - same parser as the feature-extraction notebooks (mean-mdl, kazmi-*, mean-surprisal-*),
# sentences come from the parser, so tagger, attribute_ruler, ner and lemmatizer are disabled
spacy.prefer_gpu() # uses the Mac GPU (MPS) if available
nlp = spacy.load("en_core_web_trf", disable=["tagger", "attribute_ruler", "ner", "lemmatizer"])
print(nlp.pipe_names)

/Users/adil.shamim/Desktop/Academic/code/mtech-thesis/experiment-handson/thesis/lib/python3.15/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/adil.shamim/Desktop/Academic/code/mtech-thesis/experiment-handson/thesis/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['senter']


In [2]:
def split_sentences(paragraphs):
    sentence_lists = []
    # nlp.pipe splits the paragraphs in batches - much faster than nlp(text) one by one
    for doc in nlp.pipe(paragraphs, batch_size=64):
        sentences = []
        for sent in doc.sents:
            sentence = sent.text.strip()
            if sentence: # skip empty / whitespace only sentences
               sentences.append(sentence)
        sentence_lists.append(sentences)
    return sentence_lists

In [3]:
import gzip
import json


def make_filename(article_title):
    # same file names as data-preprocessing (pipeline/utils/io.py)
    name = article_title.strip()
    for char in ["<", ">", ":", '"', "/", "\\", "|", "?", "*"]:
        name = name.replace(char, "_")
    return name + ".json"


with gzip.open(revision_groups_file, "rt", encoding="utf-8") as gz:
    for line in gz:
        article = json.loads(line)
        out_json_file = sentence_json_path / make_filename(article["article_title"])
        if out_json_file.exists() and out_json_file.stat().st_size > 0: # Skip if already exist
           continue

        # all versions of the article in one list, so the whole article is split in one nlp.pipe call
        paragraphs = []
        for revision_group in article["groups"]:
            paragraphs.extend(revision_group["versions"])
        sentence_lists = split_sentences(paragraphs)

        index = 0
        for revision_group in article["groups"]:
            new_versions = []
            for paragraph in revision_group["versions"]:
                new_versions.append({"paragraph": paragraph, "sentences": sentence_lists[index]})
                index += 1
            revision_group["versions"] = new_versions

        # write to a temp file and rename at the end, so a half written file is never skipped on restart
        tmp_json_file = out_json_file.with_suffix(".json.tmp")
        with tmp_json_file.open("w", encoding="utf-8") as of:
            json.dump(article, of, ensure_ascii=False, indent=2)
        tmp_json_file.rename(out_json_file)
#break - To demo 1 file only converted

In [4]:
# TEST SENTENCE JSON

with (sentence_json_path / "00s-Music-Quiz.json").open(encoding="utf-8") as jf:
    article = json.load(jf)
print(json.dumps(article["groups"][2]["versions"][0], ensure_ascii=False, indent=2))

print(split_sentences(["Timbaland was one of the most successful producers of the 2000s. His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake."]))

with gzip.open(revision_groups_file, "rt", encoding="utf-8") as gz:
    n_raw = sum(1 for _ in gz)
n_out = len(list(sentence_json_path.glob("*.json")))
n_tmp = len(list(sentence_json_path.glob("*.tmp")))
print("raw files:", n_raw, " sentence files:", n_out, " left over tmp:", n_tmp)
assert n_raw == n_out and n_tmp == 0

{
  "paragraph": "Do you consider yourself a 2000’s music expert? Can you belt out songs by artists like Britney Spears, Outkast, and Green Day word for word? If so, now’s your chance to show off your skills!",
  "sentences": [
    "Do you consider yourself a 2000’s music expert?",
    "Can you belt out songs by artists like Britney Spears, Outkast, and Green Day word for word?",
    "If so, now’s your chance to show off your skills!"
  ]
}
[['Timbaland was one of the most successful producers of the 2000s.', 'His biggest hits include “Promiscuous” by Nelly Furtado, “Work It” by Missy Elliott, and “Cry Me a River” by Justin Timberlake.']]
raw files: 7519  sentence files: 7519  left over tmp: 0
